# *DistilRoBERTa*

In [1]:
import os
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformer_utils import TextDataset, label_smoothing_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

LABEL2ID  = {'google': 0, 
             'anthropic': 1, 
             'meta': 2, 
             'openai': 3, 
             'human': 4}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
N_CLASSES = 5

device: cuda


## *dados*

In [2]:
df_full  = pd.read_csv('../datasets/dataset_v2_full.csv',       sep=';')
df_ex    = pd.read_csv('../datasets/dataset-exemplos.csv',      sep=';')
df_subm1 = pd.read_csv('../Subm1/subm1_labels_revealed.csv',    sep=';')

def load_xy(df):
    texts  = df['Text'].fillna('').tolist()
    labels = [LABEL2ID[l.lower()] for l in df['Label'].tolist()]
    return texts, labels

texts_synth, y_synth = load_xy(df_full)
texts_real,  y_real  = load_xy(df_subm1)
texts_val,   y_val   = load_xy(df_ex)

REAL_WEIGHT = 10
texts_train = texts_synth + texts_real * REAL_WEIGHT
y_train     = y_synth     + y_real     * REAL_WEIGHT

cw = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_real + y_val)
class_weights = torch.tensor(cw, dtype=torch.float32).to(device)

print(f'treino : {len(texts_train)} ({len(texts_synth)} sintéticos + {len(texts_real)*REAL_WEIGHT} reais×{REAL_WEIGHT})')
print(f'validação    : {len(texts_val)} reais')
print('class weights:', {ID2LABEL[i]: f'{w:.2f}' for i, w in enumerate(cw)})

treino : 6000 (5000 sintéticos + 1000 reais×10)
validação    : 125 reais
class weights: {'google': '1.36', 'anthropic': '1.12', 'meta': '1.29', 'openai': '1.45', 'human': '0.52'}


## *modelo*

In [3]:
from transformer_utils import TextDataset, label_smoothing_loss

MODEL_NAME = 'distilroberta-base'
MAX_LEN    = 128

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)


def evaluate_model(model, loader, criterion=None):
    model.eval()
    total_loss = 0.0
    total      = 0
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits  = outputs.logits

            if criterion is not None:
                total_loss += criterion(logits, labels).item() * input_ids.size(0)

            preds_all.extend(logits.argmax(dim=1).cpu().tolist())
            labels_all.extend(labels.cpu().tolist())
            total += input_ids.size(0)

    acc      = sum(p == t for p, t in zip(preds_all, labels_all)) / total
    f1       = f1_score(labels_all, preds_all, average='macro')
    avg_loss = total_loss / total if criterion is not None else None
    return avg_loss, acc, f1, preds_all

print(f'modelo: {MODEL_NAME}')

modelo: distilroberta-base


## *hiperparâmetros e DataLoaders*

In [4]:
LR           = 3e-5
LABEL_SMOOTH = 0.1
BATCH_SIZE   = 32
MAX_EPOCHS   = 15
PATIENCE     = 3
WARMUP_FRAC  = 0.1

print('a tokenizar...')
train_ds = TextDataset(texts_train, y_train, tokenizer, MAX_LEN)
val_ds   = TextDataset(texts_val,   y_val,   tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64)
print(f'tokenização concluída! | train batches: {len(train_loader)}')

a tokenizar...
tokenização concluída! | train batches: 188


## *treino*

In [ ]:
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=N_CLASSES,
    id2label=ID2LABEL, label2id=LABEL2ID
).to(device)

optimizer     = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps   = len(train_loader) * MAX_EPOCHS
warmup_steps  = int(total_steps * WARMUP_FRAC)
scheduler     = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler        = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')
val_criterion = nn.CrossEntropyLoss()

best_f1    = 0.0
best_acc   = 0.0
best_state = None
no_improve = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss  = 0.0
    running_total = 0

    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss   = label_smoothing_loss(logits, labels, N_CLASSES,
                                          smoothing=LABEL_SMOOTH, weights=class_weights)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss  += loss.item() * input_ids.size(0)
        running_total += input_ids.size(0)

    train_loss            = running_loss / running_total
    _, val_acc, val_f1, _ = evaluate_model(model, val_loader, val_criterion)

    marker = ' *' if val_f1 > best_f1 else ''
    print(f'Epoch {epoch:02d}/{MAX_EPOCHS} | train_loss={train_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f}{marker}')

    if val_f1 > best_f1:
        best_f1    = val_f1
        best_acc   = val_acc
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'early stopping no epoch {epoch}')
            break

model.load_state_dict(best_state)

os.makedirs('../models/model_bert', exist_ok=True)
model.save_pretrained('../models/model_bert')
tokenizer.save_pretrained('../models/model_bert')
print(f'\nmelhor modelo guardado -> val_acc={best_acc:.4f} | val_f1={best_f1:.4f}')

## *avaliação com dataset-exemplos*

In [6]:
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from sklearn.metrics import f1_score, classification_report
from transformer_utils import TextDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

LABEL2ID = {'google': 0, 'anthropic': 1, 'meta': 2, 'openai': 3, 'human': 4}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}

# carregar dataset-exemplos
df_ex = pd.read_csv('../datasets/dataset-exemplos.csv', sep=';')
df_ex = df_ex.dropna(subset=['Label'])
df_ex = df_ex[df_ex['Label'].str.strip().str.lower().isin(LABEL2ID.keys())]
texts_val = df_ex['Text'].fillna('').tolist()
y_val     = [LABEL2ID[l.strip().lower()] for l in df_ex['Label'].tolist()]

# carregar modelo guardado
MODEL_DIR = '../models/model_bert'
tokenizer = RobertaTokenizer.from_pretrained(MODEL_DIR)
model     = RobertaForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

# preparar loader
val_ds     = TextDataset(texts_val, y_val, tokenizer, max_len=128)
val_loader = DataLoader(val_ds, batch_size=64)

print(f'modelo carregado de {MODEL_DIR} | {len(texts_val)} exemplos de validação')

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

modelo carregado de ../models/model_bert | 125 exemplos de validação


In [7]:
_, val_acc, val_f1, val_preds = evaluate_model(model, val_loader)
print(f'[dataset-exemplos] accuracy={val_acc:.4f} | f1-macro={val_f1:.4f}')
print()
print(classification_report(
    [ID2LABEL[l] for l in y_val],
    [ID2LABEL[p] for p in val_preds],
    digits=3
))

[dataset-exemplos] accuracy=0.6720 | f1-macro=0.5841

              precision    recall  f1-score   support

   anthropic      1.000     0.304     0.467        23
      google      0.609     0.875     0.718        16
       human      0.671     0.942     0.784        52
        meta      0.588     0.588     0.588        17
      openai      0.800     0.235     0.364        17

    accuracy                          0.672       125
   macro avg      0.734     0.589     0.584       125
weighted avg      0.730     0.672     0.633       125

